<a href="https://colab.research.google.com/github/ryanrtrindade/previsao_vendas_regioes/blob/C%C3%B3digos/Previs%C3%A3o_de_vendas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Olá, o objetivo desse estudo e trabalhar com um dataset de vendas. Neste estudo iremos entender os dados, registrar insights e prever as vendas.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import datetime

In [ ]:
df_orders = pd.read_csv('/content/drive/MyDrive/VENDAS/Orders.csv')

Nesse primeiro df temos algums caracteristicas qualitativas e 2 quantitativas.
De primeira istancia, poderemos observar algumas caracteristicas deste data frame.

In [ ]:
df_orders

In [ ]:
df_orders.shape

In [ ]:
df_orders.info()

In [ ]:
df_orders.describe()

Esse segundo dataframe é responsavel pelos dados dos clientes que consomem os produtos desse fornecedor.

Faremos o mesmo procedimento e iremos entender o dataframe e verificiar se há nulos.

In [ ]:
df_clientes = pd.read_csv('/content/drive/MyDrive/VENDAS/Customers.csv')

In [ ]:
df_clientes

In [ ]:
df_clientes.shape

In [ ]:
df_clientes.info()

In [ ]:
df_clientes.describe()

In [ ]:
sns.heatmap(df_orders.isnull())

In [ ]:
df_orders.isnull().sum()

In [ ]:
df_clientes.isnull().sum()

In [ ]:
df_orders.hist( bins = 30, color = 'orange', figsize = (20,10))

um Inisght acim notado é que possuimos mais vendas totais entre 0 e 1000, e apoós isso a quantidade vai caindo gradualmente.

Outro insight é que temos mais vendas com quantidas nove, porem os valores oscilam muito e não necessiariamente quanto menor a quantidade protudos, maior será quantidade de pedidos com x número de produtos.

In [ ]:
df_clientes.hist(bins = 30, color = 'b')

No gráfico acima, vemos que possuimos mais clientes acima do 40 anos, ou seja, o público alvo é um público mais maduro.

In [ ]:
df_clientes['Age'].max()

In [ ]:
df_clientes['Age'].mean()

Iremos indexar os dataframes para prosseguir a análise.

In [ ]:
df_merged = pd.merge(df_orders, df_clientes, on = 'CustomerID')
df_merged

O propósito em dividir o dataframe entre clientes acima dos 30 e abaixo é para identificar o poder aquisitivo pela faixa etária.

In [ ]:
df_young = df_merged[df_merged['Age'] < 30]
df_young

In [ ]:
df_old = df_merged[df_merged['Age'] >= 30]
df_old

In [ ]:
df_old['CustomerID'].drop_duplicates()

In [ ]:
df_young['CustomerID'].drop_duplicates()

In [ ]:
df_young['TotalSales'].sum().mean()

In [ ]:
df_old['TotalSales'].sum().mean()

In [ ]:
print('Abaixo dos 30')
print((df_young['TotalSales'].sum()/df_merged['TotalSales'].sum())*100, '%')
print('-------------------')
print('30 ou mais')
print((df_old['TotalSales'].sum()/df_merged['TotalSales'].sum())*100,'%')

Com a análise acima, podemos concluir que os clientes mais velhos representam maior parte das vendas totais e quantidades. E podemos concluir 2 coisas:
A primeira é que possuimos mais clientes acima dos 40 e a segunda é que esses clientes possivelmente possuem um poder aquisitivo maior que os clientes abaixo dos 30.

In [ ]:
df_merged.isnull().sum()

In [ ]:
df_merged

Aqui vamos converter os dados strings em númericos para podermos construir uma matriz de correlação entre os dados visualmente.

In [ ]:
df_merged['Gender'] = df_merged['Gender'].apply(lambda x: 1 if x == 'Male' else 0)
df_merged


In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_merged[['Region', 'CustomerSince', 'OrderDate']] = df_merged[['Region', 'CustomerSince', 'OrderDate']].apply(le.fit_transform )
df_merged

In [ ]:
df_merged = df_merged.drop(['CustomerName', 'Email'], axis = 1)
df_merged

Abaixo vemos q não há tanta correlação entre os dados, porém quantidade e total de vendas estão moderadamente correlacionadas.

In [ ]:
cor = df_merged.corr()
f, ax = plt.subplots(figsize = (10,8))
sns.heatmap(cor, annot  = True)

In [ ]:
df_all = pd.merge(df_clientes, df_orders, on = 'CustomerID')
df_all

In [ ]:
df_all['Year'] = pd.to_datetime(df_all['OrderDate'], dayfirst = True).dt.year

In [ ]:
df_all['Month'] = pd.to_datetime(df_all['OrderDate'], dayfirst = True).dt.month
df_all['Day'] = pd.to_datetime(df_all['OrderDate'], dayfirst = True).dt.day

In [ ]:
df_all

In [ ]:
axis = df_all.groupby('Month')[['TotalSales']].mean().plot(figsize=(10,5), marker = 'o')
axis

In [ ]:
axis2 = df_all.groupby('Day')[['TotalSales']].mean().plot(figsize = (10, 5), marker = 'o', color = 'purple')
axis2

In [ ]:
fig, ax = plt.subplots(figsize = (20, 10))
df_all.groupby(['Age', 'Gender'])['TotalSales'].mean().unstack().plot(ax=ax)

In [ ]:
sns.barplot(x = 'Gender', y = 'TotalSales', data = df_all, hue = 'Gender' )

In [ ]:
sns.barplot(x = 'Region', y = 'TotalSales', data = df_all, hue = 'Region')

In [ ]:
!pip install prophet


In [ ]:
from prophet import Prophet

In [ ]:
def prever_gastos(region, df_all, periods):

    df_cliente = df_all[df_all['Region'] == region]
    df_cliente = df_cliente[['OrderDate', 'TotalSales']].rename(
        columns={'OrderDate': 'ds','TotalSales': 'y'})

    df_cliente['ds'] = pd.to_datetime(df_cliente['ds'])
    df_cliente = df_cliente.sort_values(by='ds')

    model = Prophet()
    model.fit(df_cliente)
    future = model.make_future_dataframe(periods=periods)
    forecast = model.predict(future)
    figure1 = model.plot(forecast, xlabel='Data', ylabel='Total Sales')
    figure2 = model.plot_components(forecast)

    return df_cliente, forecast

In [ ]:
#df = prever_gastos('West', df_all, 30)
#df

In [ ]:
df_origin, df_future = prever_gastos('West', df_all, 60)

In [ ]:
df_origin.shape, df_future.shape

In [ ]:
df_future.head()

In [ ]:
df_future.tail(10)

In [ ]:
df_origin.tail()